In [12]:
# Import tools for reading JSON data and working with file paths
import json
from pathlib import Path

# Set the expected location of the data file
input_path = "Data/intens_affect/intens_affect.json"

# Find the folder where the notebook is currently running
base_dir = Path.cwd().resolve()

# Check likely locations for the data file
candidate_paths = [
    base_dir / "Data" / "intens_affect" / "intens_affect.json",
    base_dir.parent / "Data" / "intens_affect" / "intens_affect.json",
]

# Use the first location where the data file exists
input_path = next((p for p in candidate_paths if p.exists()), candidate_paths[0])

# Open the JSON file and load its contents into Python
with open(input_path, encoding="utf-8") as f:
    data = json.load(f)

# Show how many features are in the file and the properties of the first feature
print("Number of features:", len(data["features"]))
print(data["features"][0]["properties"])

# Create an empty list to hold only economic activity areas
economic_activities = []

# Go through each feature and keep those marked as economic activities
for feature in data["features"]:
    if feature["properties"]["Affectatio"] == "Activités économiques":
        economic_activities.append(feature)

# Show how many economic activity areas were found
print("Economic activity areas:", len(economic_activities))

Number of features: 716
{'NiveauINT': 99.0, 'Niv_txt': 'Non applicable', 'Densite': 0.0, 'Affectatio': 'Conservation', 'Shape_Area': 247383.812516}
Economic activity areas: 27


In [13]:
from shapely import union_all
from shapely.geometry import shape

# Load local greenspace polygons from the matching projected dataset
base_dir = Path.cwd().resolve()
greenspace_candidates = [
    base_dir / "Data" / "affect_espacevertlocal" / "affect_espacevertlocal.json",
    base_dir.parent / "Data" / "affect_espacevertlocal" / "affect_espacevertlocal.json",
]
greenspace_path = next((p for p in greenspace_candidates if p.exists()), greenspace_candidates[0])
with open(greenspace_path, encoding="utf-8") as f:
    greenspace_data = json.load(f)

# Distances are in metres because both layers use EPSG:32188
expected_crs = "urn:ogc:def:crs:EPSG::32188"
economic_crs = data["crs"]["properties"]["name"]
greenspace_crs = greenspace_data["crs"]["properties"]["name"]
if economic_crs != expected_crs or greenspace_crs != expected_crs:
    raise ValueError("Economic areas and greenspace must both use EPSG:32188 for a 500 m distance filter.")

greenspace_union = union_all([shape(feature["geometry"]) for feature in greenspace_data["features"]])
greenspace_500m_buffer = greenspace_union.buffer(500)

# Keep economic areas that are within 500 m of at least one greenspace polygon
all_economic_activities = economic_activities.copy()
economic_activities = [
    feature
    for feature in all_economic_activities
    if shape(feature["geometry"]).intersects(greenspace_500m_buffer)
]

print("Economic activity areas within 500 m of greenspace:", len(economic_activities))

Economic activity areas within 500 m of greenspace: 24


In [14]:
# Save the economic areas within 500 m of greenspace
output_path = input_path.parent / "economic_areas_within_500m.geojson"

# Keep the original GeoJSON structure and replace its features
output_data = data.copy()
output_data["features"] = economic_activities

# Write the results to a GeoJSON file
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f)

# Confirm the output was saved correctly
print("GeoJSON saved to:", output_path)
print("Economic areas saved:", len(output_data["features"]))

# Check the saved file
with open(output_path, encoding="utf-8") as f:
    saved_data = json.load(f)

assert len(saved_data["features"]) == 24
print("Check passed: 24 economic areas saved.")

GeoJSON saved to: /Users/gianniscaramuzzi/GEOG 464/A2_Scaramuzzi/Data/intens_affect/economic_areas_within_500m.geojson
Economic areas saved: 24
Check passed: 24 economic areas saved.
